# 05j-f — Region/mechanism expert revision

05j-e localized the error to regenerative trunk and adjacent compartments. This notebook compares capacity-matched uniform experts against experts gated only by authentic teacher region and mechanism metadata. Every correction starts exactly at the frozen direct-tree prediction. Fit and calibration remain unchanged; development is evaluated only after checkpoint freeze. Held-out data and rollout remain sealed.

## 1. Coherent checkout and GPU runtime

In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKSPACE = Path('/kaggle/working/hayflow_workspace'); ELM_REPO = WORKSPACE / 'elmneuron'
if not ELM_REPO.exists(): subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'pyarrow', 'pyyaml'], check=True)
sys.path.insert(0, str(ELM_REPO)); REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Revision:', REVISION)

In [ ]:
import h5py, json, numpy as np, pandas as pd, pyarrow, torch, yaml
assert torch.cuda.is_available(), 'Attiva una GPU Kaggle prima di eseguire 05j-f.'
print({'torch': torch.__version__, 'cuda': torch.cuda.get_device_name(0)})

## 2. Exact immutable artifact chain

In [ ]:
import hashlib, shutil, zipfile
from src.hayflow_model.hines_state_normalization_repair import EXPECTED_05H_INDEX_SHA256
from src.hayflow_model.hines_netcon_semantic_repair import EXPECTED_05I_INDEX_SHA256
from src.hayflow_model.hines_synaptic_domain_repair import EXPECTED_05IB_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_recheck import EXPECTED_05IC_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_revision import EXPECTED_05J_INDEX_SHA256
from src.hayflow_model.hines_spatial_support_revision import EXPECTED_05JB_INDEX_SHA256
from src.hayflow_model.hines_trainable_topology_canary import EXPECTED_05JC_INDEX_SHA256
from src.hayflow_model.hines_architecture_reassessment import EXPECTED_05JD_INDEX_SHA256
from src.hayflow_model.hines_region_mechanism_experts import EXPECTED_05JE_INDEX_SHA256
INPUT_ROOT = Path('/kaggle/input')
def extract_zip_safely(source, destination):
    source, destination = Path(source), Path(destination); marker = destination / '.source_size'; stamp = str(source.stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp: return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True); root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve(); assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp); return destination
def index_matches(path, expected):
    path = Path(path)
    try:
        if path.is_file():
            with zipfile.ZipFile(path) as archive:
                names = [n for n in archive.namelist() if n.replace('\\', '/').endswith('artifact_index.json')]
                return len(names) == 1 and hashlib.sha256(archive.read(names[0])).hexdigest() == expected
        index = path / 'artifact_index.json'; return index.is_file() and hashlib.sha256(index.read_bytes()).hexdigest() == expected
    except (OSError, zipfile.BadZipFile): return False
def artifact(env, archive_name, marker, expected):
    candidates = ([Path(os.environ[env]).expanduser()] if os.environ.get(env) else []) + list(INPUT_ROOT.rglob(archive_name)) + [p.parent for p in INPUT_ROOT.rglob(marker)]
    valid = [p.resolve() for p in candidates if p.exists() and index_matches(p, expected)]
    assert valid, f'{archive_name} non trovato o incompatibile. Candidati: {[str(p) for p in candidates]}'
    return valid[0]
def marker_artifact(archive_name, marker):
    candidates = list(INPUT_ROOT.rglob(archive_name)) + [p.parent for p in INPUT_ROOT.rglob(marker)]
    found = next((p.resolve() for p in candidates if p.exists()), None); assert found is not None, f'{archive_name} non trovato.'; return found
topup_candidates = ([Path(os.environ['HAYFLOW_TOPUP_V3']).expanduser()] if os.environ.get('HAYFLOW_TOPUP_V3') else []) + list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip')) + [p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE = next((p.resolve() for p in topup_candidates if p.exists()), None); assert TOPUP_SOURCE is not None, 'Top-up BAP v3 non trovato.'
TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05jf_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifests = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json')); assert len(manifests) == 1, manifests; COMPOSITE_MANIFEST = manifests[0]
base_candidates = ([Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else []) + [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()] + [p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE = next((p.resolve() for p in base_candidates if p.exists()), None); assert BASE_SOURCE is not None, 'Dataset base targeted v1.1 non trovato.'
CHECKPOINT_05B_SOURCE = marker_artifact('hayflow_hines_canary_v2.zip', 'canary_models.pt')
if CHECKPOINT_05B_SOURCE.name == 'checkpoints': CHECKPOINT_05B_SOURCE = CHECKPOINT_05B_SOURCE.parent
ARTIFACT_05C_SOURCE = marker_artifact('hayflow_hines_causal_isolation.zip', 'checkpoint_forensics.json')
ARTIFACT_05D_SOURCE = marker_artifact('hayflow_hines_residual_conditioning.zip', 'free_residual_report.json')
ARTIFACT_05E_SOURCE = marker_artifact('hayflow_hines_segment_capacity.zip', 'capacity_probe_report.json')
ARTIFACT_05F_SOURCE = marker_artifact('hayflow_hines_segment_micro_canary.zip', 'micro_canary_report.json')
ARTIFACT_05G_SOURCE = marker_artifact('hayflow_hines_optimization_audit.zip', 'optimization_support.json')
ARTIFACT_05H_SOURCE = artifact('HAYFLOW_05H_ARTIFACT', 'hayflow_hines_representation_forensics.zip', 'representation_forensics_config.json', EXPECTED_05H_INDEX_SHA256)
ARTIFACT_05I_SOURCE = artifact('HAYFLOW_05I_ARTIFACT', 'hayflow_hines_state_normalization_repair.zip', 'state_normalization_repair_config.json', EXPECTED_05I_INDEX_SHA256)
ARTIFACT_05IB_SOURCE = artifact('HAYFLOW_05IB_ARTIFACT', 'hayflow_hines_netcon_semantic_state_repair.zip', 'netcon_semantic_repair_config.json', EXPECTED_05IB_INDEX_SHA256)
ARTIFACT_05IC_SOURCE = artifact('HAYFLOW_05IC_ARTIFACT', 'hayflow_hines_synaptic_domain_repair.zip', 'synaptic_domain_repair_config.json', EXPECTED_05IC_INDEX_SHA256)
ARTIFACT_05J_SOURCE = artifact('HAYFLOW_05J_ARTIFACT', 'hayflow_hines_repaired_representation_recheck.zip', 'repaired_representation_recheck_config.json', EXPECTED_05J_INDEX_SHA256)
ARTIFACT_05JB_SOURCE = artifact('HAYFLOW_05JB_ARTIFACT', 'hayflow_hines_repaired_representation_revision.zip', 'repaired_representation_revision_config.json', EXPECTED_05JB_INDEX_SHA256)
ARTIFACT_05JC_SOURCE = artifact('HAYFLOW_05JC_ARTIFACT', 'hayflow_hines_spatial_support_revision.zip', 'spatial_support_revision_config.json', EXPECTED_05JC_INDEX_SHA256)
ARTIFACT_05JD_SOURCE = artifact('HAYFLOW_05JD_ARTIFACT', 'hayflow_hines_trainable_topology_decoder_micro_canary.zip', 'trainable_topology_canary_config.json', EXPECTED_05JD_INDEX_SHA256)
ARTIFACT_05JE_SOURCE = artifact('HAYFLOW_05JE_ARTIFACT', 'hayflow_hines_architecture_reassessment.zip', 'architecture_reassessment_config.json', EXPECTED_05JE_INDEX_SHA256)
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE), '05j-d': str(ARTIFACT_05JD_SOURCE), '05j-e': str(ARTIFACT_05JE_SOURCE)})

## 3. Composite dataset preflight

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now); percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9); eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05j-f][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True); hash_last[name] = percent
bundle = prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST, base_source=BASE_SOURCE, progress=hash_progress)
display({'valid': bundle.manifest['valid'], 'fingerprint': bundle.fingerprint, 'transition_count': bundle.transition_count})
assert bundle.manifest['valid'] and bundle.transition_count == 29880 and not bundle.manifest['physical_merge_performed']

## 4. Session and exact 05j-e diagnosis

In [ ]:
from src.hayflow_model import HinesArchitectureReassessmentConfig, HinesCapacityConfig, HinesConditioningConfig, HinesIsolationConfig, HinesNetConSemanticRepairConfig, HinesOptimizationAuditConfig, HinesPrototypeExperimentConfig, HinesRegionMechanismExpertConfig, HinesRegionMechanismExpertRevision, HinesRepairedRepresentationRecheckConfig, HinesRepairedRepresentationRevisionConfig, HinesRepresentationForensicsConfig, HinesSegmentCanaryConfig, HinesSpatialSupportRevisionConfig, HinesStateNormalizationRepairConfig, HinesSynapticDomainRepairConfig, HinesTrainableTopologyCanaryConfig
def config(name): return yaml.safe_load((ELM_REPO / 'configs/hayflow' / name).read_text())
base_config=config('hayflow_hines_optimization_audit.yml'); forensic=config('hayflow_hines_representation_forensics.yml'); repair=config('hayflow_hines_state_normalization_repair.yml'); netcon=config('hayflow_hines_netcon_semantic_repair.yml'); domain=config('hayflow_hines_synaptic_domain_repair.yml'); recheck=config('hayflow_hines_repaired_representation_recheck.yml'); revision=config('hayflow_hines_repaired_representation_revision.yml'); spatial=config('hayflow_hines_spatial_support_revision.yml'); topology=config('hayflow_hines_trainable_topology_canary.yml'); reassessment=config('hayflow_hines_architecture_reassessment.yml'); expert_payload=config('hayflow_hines_region_mechanism_experts.yml')
model_config=HinesPrototypeExperimentConfig.from_mapping(base_config['model_experiment']); isolation_config=HinesIsolationConfig.from_mapping(base_config['isolation']); conditioning_config=HinesConditioningConfig.from_mapping(base_config['conditioning']); capacity_config=HinesCapacityConfig.from_mapping(base_config['capacity']); canary_config=HinesSegmentCanaryConfig.from_mapping(base_config['micro_canary']); audit_config=HinesOptimizationAuditConfig.from_mapping(base_config['optimization_audit']); representation_config=HinesRepresentationForensicsConfig.from_mapping(forensic['representation_forensics']); repair_config=HinesStateNormalizationRepairConfig.from_mapping(repair['state_normalization_repair']); netcon_config=HinesNetConSemanticRepairConfig.from_mapping(netcon['netcon_semantic_repair']); domain_config=HinesSynapticDomainRepairConfig.from_mapping(domain['synaptic_domain_repair']); recheck_config=HinesRepairedRepresentationRecheckConfig.from_mapping(recheck['repaired_representation_recheck']); revision_config=HinesRepairedRepresentationRevisionConfig.from_mapping(revision['repaired_representation_revision']); spatial_config=HinesSpatialSupportRevisionConfig.from_mapping(spatial['spatial_support_revision']); topology_config=HinesTrainableTopologyCanaryConfig.from_mapping(topology['trainable_topology_canary']); reassessment_config=HinesArchitectureReassessmentConfig.from_mapping(reassessment['architecture_reassessment']); expert_config=HinesRegionMechanismExpertConfig.from_mapping(expert_payload['region_mechanism_experts'])
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_region_mechanism_expert_revision')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
session = HinesRegionMechanismExpertRevision(bundle, OUTPUT_DIR, model_config, isolation_config, conditioning_config, capacity_config, canary_config, audit_config, representation_config, CHECKPOINT_05B_SOURCE, ARTIFACT_05C_SOURCE, ARTIFACT_05D_SOURCE, ARTIFACT_05E_SOURCE, ARTIFACT_05F_SOURCE, ARTIFACT_05G_SOURCE, repair_config=repair_config, artifact_05h_source=ARTIFACT_05H_SOURCE, netcon_config=netcon_config, artifact_05i_source=ARTIFACT_05I_SOURCE, domain_config=domain_config, artifact_05ib_source=ARTIFACT_05IB_SOURCE, recheck_config=recheck_config, artifact_05ic_source=ARTIFACT_05IC_SOURCE, revision_config=revision_config, artifact_05j_source=ARTIFACT_05J_SOURCE, spatial_config=spatial_config, artifact_05jb_source=ARTIFACT_05JB_SOURCE, topology_config=topology_config, artifact_05jc_source=ARTIFACT_05JC_SOURCE, reassessment_config=reassessment_config, artifact_05jd_source=ARTIFACT_05JD_SOURCE, expert_config=expert_config, artifact_05je_source=ARTIFACT_05JE_SOURCE, code_revision=REVISION)
prepare_report = session.prepare_region_mechanism_expert_revision()
display({'revision': REVISION, '05j-e': prepare_report['artifact_05je'], 'comparison': prepare_report['comparison']})
assert expert_config.families == ('uniform_expert_control', 'region_mechanism_experts')
assert prepare_report['expert_definition_roles'] == ['teacher_metadata_only']
assert not prepare_report['development_used_for_model_selection'] and not prepare_report['heldout_inputs_extracted'] and not prepare_report['rollout_performed']

## 5. Rebuild the unchanged 36/12 design and frozen direct-tree predictions

In [ ]:
normalizer_report=session.apply_verified_synaptic_domain_normalizer(); support_report=session.build_expanded_train_support(); feature_report=session.prepare_expanded_spatial_features(); design_report=session.prepare_topology_canary_designs(); ridge_report=session.fit_fixed_tree_ridge_baseline(); reconstruction_report=session.reconstruct_frozen_checkpoints(metric_atol=expert_config.checkpoint_reconstruction_metric_atol)
display({'design_valid': design_report['valid'], 'fit_pairs': design_report['fit_pair_count'], 'calibration_pairs': design_report['calibration_pair_count'], 'reconstruction_valid': reconstruction_report['valid']})
assert design_report['valid'] and reconstruction_report['valid']
assert reconstruction_report['metric_atol'] == 0.0002 and reconstruction_report['explicit_metric_atol_override']
assert not reconstruction_report['retraining_performed'] and not reconstruction_report['development_used_for_model_selection']

## 6. Target-independent expert masks

In [ ]:
gate_report = session.prepare_expert_gates()
display(gate_report)
assert gate_report['valid'] and gate_report['all_gate_rows_sum_to_one']
assert not gate_report['target_values_used'] and not gate_report['development_values_used']

## 7. Capacity-matched expert canary over three seeds

In [ ]:
canary_report = session.run_region_mechanism_expert_canary()
display(pd.DataFrame(canary_report['family_summary']))
assert canary_report['valid'] and not canary_report['development_used_for_checkpoint_selection']
assert canary_report['development_inference_after_checkpoint_freeze']
assert not canary_report['heldout_inputs_extracted'] and not canary_report['rollout_performed']

## 8. Scoped decision

In [ ]:
final_report = session.finalize_region_mechanism_expert_revision(gate_report, canary_report)
display({'valid': final_report['valid'], 'diagnosis': final_report['diagnosis'], 'expert_vs_uniform': final_report['expert_vs_uniform'], 'passing_families': final_report['passing_families'], 'next_step': final_report['next_step']})
assert final_report['valid'] and not final_report['full_training_authorized']
assert not final_report['methodology']['development_used_for_checkpoint_selection']
assert not final_report['methodology']['heldout_inputs_extracted']
assert not final_report['methodology']['rollout_performed']

## 9. Create and download the ZIP

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, display
zip_base = Path('/kaggle/working/hayflow_hines_region_mechanism_expert_revision')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
payload = base64.b64encode(zip_path.read_bytes()).decode('ascii'); filename = zip_path.name
display(Javascript(f"""
const binary = atob('{payload}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
"""))
print({'zip': str(zip_path), 'size_mib': round(zip_path.stat().st_size / 2**20, 2), 'download': 'avviato dal browser'})